# 03 マクロ分析: 言語別トレンド

Stack Overflow 質問データと Developer Survey を組み合わせ、
Python / JavaScript / Java / Go の「盛衰」を多角的に可視化する。

**出力ファイル**
- `data/analysis/trends.parquet` — 内部集計結果
- `web/static/data/trends.json` — フロントエンド用 JSON

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from config.settings import PROCESSED_DIR, ANALYSIS_DIR, ROOT_DIR
from src.api.survey_loader import SurveyLoader
from src.analysis.trend import TrendAnalyzer

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print('設定完了')

## 1. データ読み込み

In [ ]:
# Stack Overflow 質問データ
questions_df = pd.read_parquet(PROCESSED_DIR / 'questions.parquet')
print(f'質問数: {len(questions_df):,}')
print(f'言語: {questions_df["language"].unique()}')
print(f'年: {sorted(questions_df["year"].unique())}')

In [ ]:
# タグ統計
tags_df = pd.read_parquet(PROCESSED_DIR / 'tags.parquet')
print(f'タグ数: {len(tags_df)}')
tags_df.head(10)

In [ ]:
# Developer Survey（2022〜2024年）
loader = SurveyLoader()
survey_dfs: dict[int, pd.DataFrame] = {}
for year in [2022, 2023, 2024]:
    try:
        df = loader.load_year(year)
        survey_dfs[year] = df
        print(f'{year}: {len(df):,} 行, {df.shape[1]} 列')
    except FileNotFoundError as e:
        print(f'{year}: ファイルなし — {e}')

## 2. マクロ分析実行

In [ ]:
analyzer = TrendAnalyzer()

# Developer Survey から使用率・希望率を集計
survey_trend = analyzer.compute_survey_trend(survey_dfs)
print(f'\n集計行数: {len(survey_trend)}')
survey_trend

In [ ]:
# 質問サンプルから年別指標を集計
yearly_counts = analyzer.compute_yearly_counts(questions_df)
print(f'集計行数: {len(yearly_counts)}')
yearly_counts

In [ ]:
# 両者を year × language で結合
merged = analyzer.merge_survey_data(yearly_counts, survey_trend)
merged.head(10)

## 3. 可視化

In [ ]:
# Developer Survey 使用率の推移
if not survey_trend.empty:
    fig = px.line(
        survey_trend,
        x='year', y='usage_pct', color='display_name',
        markers=True,
        title='言語別 使用率推移（Stack Overflow Developer Survey）',
        labels={'usage_pct': '使用率 (%)', 'year': '年', 'display_name': '言語'},
        color_discrete_map={
            'Python': '#3776AB',
            'JavaScript': '#F7DF1E',
            'Java': '#ED8B00',
            'Go': '#00ADD8',
        }
    )
    fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))
    fig.show()
else:
    print('survey_trend が空です')

In [ ]:
# 希望率（学習需要）の推移
if not survey_trend.empty and survey_trend['want_pct'].notna().any():
    fig2 = px.line(
        survey_trend,
        x='year', y='want_pct', color='display_name',
        markers=True,
        title='言語別 習得希望率推移（次に使いたい言語）',
        labels={'want_pct': '習得希望率 (%)', 'year': '年', 'display_name': '言語'},
        color_discrete_map={
            'Python': '#3776AB',
            'JavaScript': '#F7DF1E',
            'Java': '#ED8B00',
            'Go': '#00ADD8',
        }
    )
    fig2.update_layout(xaxis=dict(tickmode='linear', dtick=1))
    fig2.show()

In [ ]:
# 質問スコアの中央値（言語別）
fig3 = px.bar(
    yearly_counts.groupby('language')['median_score'].mean().reset_index(),
    x='language', y='median_score',
    title='言語別 質問スコア中央値（全期間平均）',
    labels={'language': '言語', 'median_score': 'スコア中央値'},
    color='language',
)
fig3.show()

In [ ]:
# 回答率（言語別）
fig4 = px.bar(
    yearly_counts.groupby('language')['answer_rate'].mean().reset_index(),
    x='language', y='answer_rate',
    title='言語別 回答率（全期間平均）',
    labels={'language': '言語', 'answer_rate': '回答率'},
    color='language',
)
fig4.update_layout(yaxis_tickformat='.0%')
fig4.show()

In [ ]:
# Aspiration Index（学習需要 ÷ 現使用率）
if 'aspiration' in survey_trend.columns and survey_trend['aspiration'].notna().any():
    fig5 = px.bar(
        survey_trend.dropna(subset=['aspiration']),
        x='year', y='aspiration', color='display_name',
        barmode='group',
        title='Aspiration Index（習得希望率 ÷ 使用率）',
        labels={'aspiration': 'Aspiration Index', 'year': '年', 'display_name': '言語'},
    )
    fig5.add_hline(y=1.0, line_dash='dash', line_color='gray',
                   annotation_text='需要＝供給の均衡ライン')
    fig5.show()

## 4. データ保存

In [ ]:
# trends.parquet を保存
out_parquet = ANALYSIS_DIR / 'trends.parquet'
merged.to_parquet(out_parquet, index=False)
print(f'保存: {out_parquet}')

In [ ]:
# フロントエンド用 trends.json を生成
out_json = ROOT_DIR / 'web' / 'static' / 'data' / 'trends.json'
analyzer.build_web_json(survey_trend, tags_df, out_json)
print(f'JSON 生成完了: {out_json}')

In [ ]:
# 生成した JSON を確認
import json
with open(out_json) as f:
    payload = json.load(f)

print('years:', payload['years'])
for lang, info in payload['languages'].items():
    print(f"  {lang}: 使用率={info['usage_pct']}, 総質問数={info['total_questions']:,}")